In [2]:
import sqlite3
import pandas as pd

con = sqlite3.connect(r"C:\Users\DELL\opslens\data\opslens.db")

def q(sql):
    """Run a SQL query and return the result as a table."""
    return pd.read_sql(sql, con)

pd.set_option("display.max_columns", 50)
print("connected")

connected


In [3]:
q("""
SELECT
    COUNT(*)                                                       AS tickets,
    COUNT(DISTINCT user_id)                                        AS users,
    ROUND(AVG(CASE WHEN status='resolved' THEN 1.0 ELSE 0 END), 4) AS resolution_rate,
    ROUND(AVG(repeat_contact * 1.0), 4)                            AS repeat_contact_rate,
    ROUND(AVG(sla_breached * 1.0), 4)                              AS sla_breach_rate,
    ROUND(AVG(resolution_hours), 2)                                AS mean_resolution_hours,
    ROUND(AVG(csat), 3)                                            AS avg_csat
FROM clean_tickets
""")

,tickets,users,resolution_rate,repeat_contact_rate,sla_breach_rate,mean_resolution_hours,avg_csat
0,34307,13180,0.9383,0.1838,0.0855,15.28,4.071


In [4]:
q("""
SELECT
    product_area,
    COUNT(*)                                           AS tickets,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_volume,
    SUM(sla_breached)                                  AS sla_breaches_absolute,
    ROUND(AVG(sla_breached * 1.0), 4)                  AS sla_breach_rate,
    ROUND(AVG(repeat_contact * 1.0), 4)                AS repeat_contact_rate,
    ROUND(AVG(resolution_hours), 2)                    AS mean_resolution_hours,
    ROUND(AVG(csat), 3)                                AS avg_csat
FROM clean_tickets
GROUP BY product_area
ORDER BY tickets DESC
""")

,product_area,tickets,pct_of_volume,sla_breaches_absolute,sla_breach_rate,repeat_contact_rate,mean_resolution_hours,avg_csat
0,Payments,8980,26.2,535,0.0596,0.1722,11.58,4.100
1,Orders,6537,19.1,238,0.0364,0.1432,11.55,4.151
2,Delivery,5796,16.9,968,0.1670,0.1860,22.83,4.009
3,Refunds,5508,16.1,1033,0.1875,0.3103,25.69,3.875
4,Account,4549,13.3,82,0.0180,0.1409,9.04,4.149
5,Subscription,2937,8.6,76,0.0259,0.1352,10.33,4.168


In [5]:
q("""
WITH area_metrics AS (
    SELECT product_area,
           COUNT(*)                  AS tickets,
           AVG(repeat_contact * 1.0) AS repeat_rate,
           AVG(sla_breached * 1.0)   AS breach_rate,
           AVG(resolution_hours)     AS mean_hours,
           AVG(csat)                 AS avg_csat
    FROM clean_tickets
    GROUP BY product_area
)
SELECT product_area, tickets,
       RANK() OVER (ORDER BY tickets     DESC) AS rank_by_volume,
       RANK() OVER (ORDER BY repeat_rate DESC) AS rank_by_repeat,
       RANK() OVER (ORDER BY breach_rate DESC) AS rank_by_breach,
       RANK() OVER (ORDER BY mean_hours  DESC) AS rank_by_slowness,
       RANK() OVER (ORDER BY avg_csat    ASC)  AS rank_by_worst_csat
FROM area_metrics
ORDER BY rank_by_repeat
""")

,product_area,tickets,rank_by_volume,rank_by_repeat,rank_by_breach,rank_by_slowness,rank_by_worst_csat
0,Refunds,5508,4,1,1,1,1
1,Delivery,5796,3,2,2,2,2
2,Payments,8980,1,3,3,3,3
3,Orders,6537,2,4,4,4,5
4,Account,4549,5,5,6,6,4
5,Subscription,2937,6,6,5,5,6


In [6]:
q("""
WITH baseline AS (
    SELECT AVG(repeat_contact * 1.0) AS overall_repeat_rate
    FROM clean_tickets
)
SELECT
    t.issue_type,
    COUNT(*)                                                      AS tickets,
    ROUND(AVG(t.repeat_contact * 1.0), 4)                         AS repeat_rate,
    ROUND(AVG(t.repeat_contact * 1.0) / b.overall_repeat_rate, 2) AS lift_vs_baseline,
    ROUND(AVG(t.resolution_hours), 1)                             AS mean_hours,
    ROUND(AVG(t.csat), 2)                                         AS avg_csat,
    ROUND(COUNT(*) * (AVG(t.repeat_contact * 1.0)
                      - b.overall_repeat_rate))                   AS excess_contacts
FROM clean_tickets t
CROSS JOIN baseline b
WHERE t.product_area = 'Refunds'
GROUP BY t.issue_type, b.overall_repeat_rate
ORDER BY excess_contacts DESC
""")

,issue_type,tickets,repeat_rate,lift_vs_baseline,mean_hours,avg_csat,excess_contacts
0,refund_status_unclear,1871,0.3565,1.94,27.7,3.80,323.0
1,refund_not_received,1982,0.3441,1.87,31.1,3.80,318.0
2,partial_refund,875,0.2229,1.21,17.1,4.05,34.0
3,refund_rejected,780,0.2115,1.15,17.0,4.04,22.0


In [7]:
q("""
WITH baseline AS (
    SELECT AVG(repeat_contact * 1.0) AS overall_repeat_rate
    FROM clean_tickets
)
SELECT
    t.issue_type,
    COUNT(*)                                                      AS tickets,
    ROUND(AVG(t.repeat_contact * 1.0), 4)                         AS repeat_rate,
    ROUND(AVG(t.repeat_contact * 1.0) / b.overall_repeat_rate, 2) AS lift_vs_baseline,
    ROUND(AVG(t.resolution_hours), 1)                             AS mean_hours,
    ROUND(AVG(t.csat), 2)                                         AS avg_csat,
    ROUND(COUNT(*) * (AVG(t.repeat_contact * 1.0)
                      - b.overall_repeat_rate))                   AS excess_contacts
FROM clean_tickets t
CROSS JOIN baseline b
WHERE t.product_area = 'Refunds'
GROUP BY t.issue_type, b.overall_repeat_rate
ORDER BY excess_contacts DESC
""")

,issue_type,tickets,repeat_rate,lift_vs_baseline,mean_hours,avg_csat,excess_contacts
0,refund_status_unclear,1871,0.3565,1.94,27.7,3.80,323.0
1,refund_not_received,1982,0.3441,1.87,31.1,3.80,318.0
2,partial_refund,875,0.2229,1.21,17.1,4.05,34.0
3,refund_rejected,780,0.2115,1.15,17.0,4.04,22.0


In [8]:
q("""
WITH segment_rates AS (
    SELECT
        user_segment,
        AVG(CASE WHEN product_area =  'Refunds' THEN repeat_contact * 1.0 END) AS refunds_repeat_rate,
        AVG(CASE WHEN product_area <> 'Refunds' THEN repeat_contact * 1.0 END) AS other_repeat_rate,
        SUM(CASE WHEN product_area =  'Refunds' THEN 1 ELSE 0 END)             AS refunds_tickets
    FROM clean_tickets
    GROUP BY user_segment
)
SELECT
    user_segment,
    refunds_tickets,
    ROUND(refunds_repeat_rate, 4)                     AS refunds_repeat_rate,
    ROUND(other_repeat_rate, 4)                       AS elsewhere_repeat_rate,
    ROUND(refunds_repeat_rate - other_repeat_rate, 4) AS friction_gap
FROM segment_rates
WHERE refunds_tickets > 100
ORDER BY friction_gap DESC
""")

,user_segment,refunds_tickets,refunds_repeat_rate,elsewhere_repeat_rate,friction_gap
0,Enterprise,530,0.2811,0.1190,0.1621
1,Consumer,2952,0.3012,0.1442,0.1569
2,VIP,369,0.2602,0.1113,0.1488
3,SMB,1598,0.3442,0.2102,0.1340


In [9]:
q("""
SELECT
    month,
    SUM(CASE WHEN product_area = 'Refunds' THEN 1 ELSE 0 END)  AS refunds_tickets,
    ROUND(AVG(CASE WHEN product_area =  'Refunds'
                   THEN repeat_contact * 1.0 END), 4)          AS refunds_repeat_rate,
    ROUND(AVG(CASE WHEN product_area <> 'Refunds'
                   THEN repeat_contact * 1.0 END), 4)          AS control_repeat_rate,
    ROUND(AVG(CASE WHEN product_area = 'Refunds'
                   THEN resolution_hours END), 1)              AS refunds_mean_hours
FROM clean_tickets
GROUP BY month
ORDER BY month
""")

,month,refunds_tickets,refunds_repeat_rate,control_repeat_rate,refunds_mean_hours
0,2026-03,785,0.2535,0.1354,23.5
1,2026-04,788,0.2792,0.1618,22.5
2,2026-05,861,0.2927,0.1646,23.3
3,2026-06,971,0.3234,0.1661,26.7
4,2026-07,1044,0.3333,0.1647,29.6
5,2026-08,1059,0.3551,0.1619,27.0


In [10]:
q("""
WITH ordered AS (
    SELECT
        ticket_id,
        product_area,
        created_at,
        repeat_contact AS flag_in_data,
        LAG(created_at) OVER (
            PARTITION BY user_id, product_area
            ORDER BY created_at
        ) AS previous_contact_at
    FROM clean_tickets
),
derived AS (
    SELECT
        flag_in_data,
        CASE
            WHEN previous_contact_at IS NOT NULL
             AND julianday(REPLACE(created_at, 'T', ' '))
               - julianday(REPLACE(previous_contact_at, 'T', ' ')) <= 7
            THEN 1 ELSE 0
        END AS flag_derived
    FROM ordered
)
SELECT
    COUNT(*)                                                 AS tickets,
    SUM(flag_in_data)                                        AS flagged_in_data,
    SUM(flag_derived)                                        AS flagged_by_our_rule,
    ROUND(100.0 * SUM(CASE WHEN flag_in_data = flag_derived
                           THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_agreement
FROM derived
""")

,tickets,flagged_in_data,flagged_by_our_rule,pct_agreement
0,34307,6307,6672,98.94


In [11]:
import re
from collections import Counter

focus = q("""
SELECT ticket_id, issue_type, customer_text, repeat_contact, created_at
FROM clean_tickets
WHERE issue_type IN ('refund_status_unclear', 'refund_not_received')
""")

print(f"{len(focus):,} tickets to analyse\n")

STOPWORDS = set("""i my me the a an and or of to for it is was be been on in at this
that they you your we our have has had not no with but so as still are do does did
can cannot am if what when where why how them there here""".split())

words = Counter()
for text in focus["customer_text"].str.lower():
    words.update(w for w in re.findall(r"[a-z]{3,}", text) if w not in STOPWORDS)

pd.DataFrame(words.most_common(20), columns=["word", "count"])

3,853 tickets to analyse



,word,count
0,refund,2649
1,days,1586
2,ord,952
3,pending,798
4,again,730
5,nobody,626
6,tell,592
7,give,569
8,date,569
9,processing,548


In [12]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")

client = OpenAI(
    base_url=os.getenv("LLM_BASE_URL"),
    api_key=os.getenv("LLM_API_KEY"),
)

response = client.chat.completions.create(
    model=os.getenv("LLM_MODEL"),
    messages=[{"role": "user", "content": "Reply with exactly: connection works"}],
    max_tokens=20,
)

print(response.choices[0].message.content)

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [13]:
import os
from dotenv import load_dotenv

print("notebook is in:", os.getcwd())
print("parent folder contains:", os.listdir(".."))
print()

loaded = load_dotenv("../.env")
print("load_dotenv found the file:", loaded)
print("LLM_BASE_URL:", os.getenv("LLM_BASE_URL"))
print("LLM_API_KEY:", (os.getenv("LLM_API_KEY") or "NONE")[:8])

notebook is in: C:\Users\DELL\opslens
parent folder contains: ['.anaconda', '.claude', '.claude.json', '.conda', '.continuum', '.copilot', '.gemini', '.gitconfig', '.ipynb_checkpoints', '.ipython', '.jupyter', '.local', '.matplotlib', '.ms-ad', '.vscode', '.vscode-shared', '3D Objects', 'anaconda3', 'anaconda_projects', 'AppData', 'Application Data', 'clinics.csv', 'Contacts', 'Cookies', 'data cleaning methods.ipynb', 'datasetpractice.ipynb', 'Desktop', 'Documents', 'Downloads', 'Favorites', 'final.ipynb', 'firsteda.ipynb', 'for practice', 'GROUPBY AND AGG FUNCTIONS.ipynb', 'INDEXING BY ALEX.ipynb', 'IntelGraphicsProfiles', 'job-fit-analyser', 'joins merge and concat.ipynb', 'learning1.ipynb', 'Links', 'Local Settings', 'marketing-ab-test', 'Microsoft', 'monthly_targets.csv', 'Music', 'My Documents', 'NetHood', 'New folder', 'neweee', 'news', 'NTUSER.DAT', 'ntuser.dat.LOG1', 'ntuser.dat.LOG2', 'NTUSER.DAT{2ad838bc-efea-11ee-a54d-000d3a94eaa1}.TM.blf', 'NTUSER.DAT{2ad838bc-efea-11ee-a54

In [15]:
import os
from dotenv import load_dotenv

print("here:", os.getcwd())
print(".env exists here:", os.path.exists(".env"))
print("db exists here:", os.path.exists("data/opslens.db"))

here: C:\Users\DELL\opslens
.env exists here: True
db exists here: True


In [16]:
load_dotenv(".env", override=True)

print("URL:", os.getenv("LLM_BASE_URL"))
print("key starts:", (os.getenv("LLM_API_KEY") or "NONE")[:8])

URL: https://api.groq.com/openai/v1
key starts: gsk_REDACTED


In [17]:
from openai import OpenAI

client = OpenAI(
    base_url=os.getenv("LLM_BASE_URL"),
    api_key=os.getenv("LLM_API_KEY"),
)

response = client.chat.completions.create(
    model=os.getenv("LLM_MODEL"),
    messages=[{"role": "user", "content": "Reply with exactly: connection works"}],
    max_tokens=20,
)

print(response.choices[0].message.content)

NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

In [18]:
models = sorted(m.id for m in client.models.list().data)
for m in models:
    print(m)

allam-2-7b
canopylabs/orpheus-arabic-saudi
canopylabs/orpheus-v1-english
groq/compound
groq/compound-mini
meta-llama/llama-prompt-guard-2-22m
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-120b
openai/gpt-oss-20b
openai/gpt-oss-safeguard-20b
qwen/qwen3.6-27b
qwen/qwen3.8-27b
whisper-large-v3
whisper-large-v3-turbo


In [20]:
MODEL = "openai/gpt-oss-120b"

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Reply with exactly: connection works"}],
    max_tokens=20,
)

print(response.choices[0].message.content)

In [21]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "Reply with exactly: connection works"}],
    max_tokens=300,
)

msg = response.choices[0].message
print("CONTENT:", repr(msg.content))
print("FINISH REASON:", response.choices[0].finish_reason)
print("TOKENS USED:", response.usage.completion_tokens)

CONTENT: 'connection works'
FINISH REASON: stop
TOKENS USED: 39


In [22]:
import numpy as np

focus = q("""
SELECT ticket_id, issue_type, customer_text, repeat_contact, created_at, user_segment
FROM clean_tickets
WHERE issue_type IN ('refund_status_unclear', 'refund_not_received')
""")

focus["created_at"] = pd.to_datetime(focus["created_at"], format="mixed")
focus["period"] = np.where(focus["created_at"] < "2026-06-15", "pre", "post")

# 50 tickets from each issue_type x period cell = 200 total
sample = (focus
          .groupby(["issue_type", "period"], group_keys=False)
          .apply(lambda g: g.sample(50, random_state=42)))

print(f"sampled {len(sample)} of {len(focus)} tickets\n")
print(sample.groupby(["issue_type", "period"]).size())

sampled 200 of 3853 tickets



KeyError: 'issue_type'

In [23]:
import numpy as np

focus = q("""
SELECT ticket_id, issue_type, customer_text, repeat_contact, created_at, user_segment
FROM clean_tickets
WHERE issue_type IN ('refund_status_unclear', 'refund_not_received')
""")

focus["created_at"] = pd.to_datetime(focus["created_at"], format="mixed")
focus["period"] = np.where(focus["created_at"] < "2026-06-15", "pre", "post")

# 50 tickets from each issue_type x period cell = 200 total
sample = focus.groupby(["issue_type", "period"]).sample(50, random_state=42)

print(f"sampled {len(sample)} of {len(focus)} tickets\n")
print(sample.groupby(["issue_type", "period"]).size())

sampled 200 of 3853 tickets

issue_type             period
refund_not_received    post      50
                       pre       50
refund_status_unclear  post      50
                       pre       50
dtype: int64


In [24]:
import json

THEMES = [
    "no_status_visibility",      # cannot see where the refund is
    "timeline_not_met",          # promised date passed
    "no_response_from_support",  # chasing, nobody replied
    "money_not_received",        # refund confirmed but funds absent
    "amount_incorrect",          # wrong amount refunded
    "other",
]

SYSTEM_PROMPT = f"""You analyse customer support tickets for a product team.

Classify the ticket into EXACTLY ONE theme from this list:
{json.dumps(THEMES, indent=2)}

Return ONLY a JSON object, no prose, no markdown fences:

{{
  "theme": "<one theme from the list>",
  "pain_point": "<what the customer cannot do, max 12 words>",
  "severity": "<low|medium|high>",
  "root_cause_hypothesis": "<a PRODUCT-side explanation, max 15 words>",
  "self_service_gap": <true if the customer contacted support only to obtain
                       information the product should have shown them>
}}

Rules:
- theme must match the list exactly
- root_cause_hypothesis is a hypothesis, not a fact
- if unsure, use "other" rather than guessing
"""

def classify(text, model=None):
    """Classify one ticket. Returns a dict, or None if the model output was unusable."""
    response = client.chat.completions.create(
        model=model or os.getenv("LLM_MODEL"),
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": text},
        ],
        max_tokens=1000,      # reasoning tokens count toward this
        temperature=0,        # deterministic: same input, same output
    )
    raw = response.choices[0].message.content or ""
    raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```")
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return None


# test on one ticket first
test_ticket = sample.iloc[0]
print("TEXT:", test_ticket["customer_text"])
print()
print("RESULT:", classify(test_ticket["customer_text"]))

TEXT: Following up again. Still not credited after 7 days. I am out of pocket and nobody will give me a date.



NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

In [25]:
with open(".env", "r", encoding="utf-8") as f:
    text = f.read()

text = text.replace("llama-3.3-70b-versatile", "openai/gpt-oss-120b")

with open(".env", "w", encoding="utf-8") as f:
    f.write(text)

load_dotenv(".env", override=True)
print("model is now:", os.getenv("LLM_MODEL"))

model is now: openai/gpt-oss-120b


In [26]:
test_ticket = sample.iloc[0]
print("TEXT:", test_ticket["customer_text"])
print()
print("RESULT:", classify(test_ticket["customer_text"]))

TEXT: Following up again. Still not credited after 7 days. I am out of pocket and nobody will give me a date.

RESULT: {'theme': 'money_not_received', 'pain_point': 'cannot access credited funds', 'severity': 'high', 'root_cause_hypothesis': 'internal payout processing delay due to verification backlog', 'self_service_gap': True}


In [27]:
import time

def classify_with_retry(text, attempts=3):
    """Classify one ticket, retrying on transient errors. Returns (result, error)."""
    for attempt in range(attempts):
        try:
            result = classify(text)
            if result is None:
                return None, "unparseable_json"
            return result, None
        except Exception as e:
            error_name = type(e).__name__
            if attempt < attempts - 1:
                time.sleep(2 ** attempt)   # 1s, 2s, 4s
                continue
            return None, error_name
    return None, "exhausted_retries"


results, failures = [], []

for i, row in enumerate(sample.itertuples(), start=1):
    result, error = classify_with_retry(row.customer_text)

    if result is None:
        failures.append({"ticket_id": row.ticket_id, "error": error})
    else:
        result["ticket_id"] = row.ticket_id
        result["issue_type"] = row.issue_type
        result["period"] = row.period
        results.append(result)

    if i % 25 == 0:
        print(f"{i}/{len(sample)}  ok={len(results)}  failed={len(failures)}")
    time.sleep(2.2)   # stay under 30 requests/minute

ai = pd.DataFrame(results)
print(f"\nDONE  classified={len(ai)}  failed={len(failures)}")
print(f"success rate: {len(ai) / len(sample):.1%}")

25/200  ok=25  failed=0
50/200  ok=50  failed=0
75/200  ok=75  failed=0
100/200  ok=100  failed=0
125/200  ok=125  failed=0
150/200  ok=150  failed=0
175/200  ok=175  failed=0
200/200  ok=200  failed=0

DONE  classified=200  failed=0
success rate: 100.0%


In [28]:
ai.to_csv("data/ai_classifications.csv", index=False)
print("saved", len(ai), "rows")

saved 200 rows


In [29]:
theme_counts = (ai["theme"]
                .value_counts()
                .rename_axis("theme")
                .reset_index(name="tickets"))
theme_counts["pct"] = (100 * theme_counts["tickets"] / len(ai)).round(1)

print(theme_counts.to_string(index=False))
print()
print("self_service_gap = True:", f"{ai['self_service_gap'].mean():.1%}")
print()
print(pd.crosstab(ai["theme"], ai["severity"]))

                   theme  tickets  pct
    no_status_visibility       85 42.5
      money_not_received       84 42.0
        timeline_not_met       26 13.0
no_response_from_support        5  2.5

self_service_gap = True: 94.0%

severity                  high  medium
theme                                 
money_not_received          74      10
no_response_from_support     5       0
no_status_visibility        16      69
timeline_not_met            19       7


In [30]:
val = ai.sample(30, random_state=1)[["ticket_id", "theme", "self_service_gap"]]
val = val.rename(columns={"theme": "ai_theme", "self_service_gap": "ai_gap"})

# attach the original text
val = val.merge(sample[["ticket_id", "customer_text"]], on="ticket_id")

print("THEMES:", THEMES)
print()
for i, row in enumerate(val.itertuples(), start=1):
    print(f"{i:2}. {row.customer_text}")

THEMES: ['no_status_visibility', 'timeline_not_met', 'no_response_from_support', 'money_not_received', 'amount_incorrect', 'other']

 1. I was told 5-7 working days for the refund on ORD-670569. We are well past that.
 2. Chasing this again. Refund was approved but nothing has arrived in my account.
 3. no refund yet. 6 days. please sort this
 4. I keep having to chase this. There is zero visibility on where my refund is.
 5. The refnd page still says processing. It has said processing for 2 days.
 6. Contacting again because I have had no update at all. The status never changes.
 7. This is my second time asking about the refund for ORD-121931. Nobody can tell me when I get my money back.
 8. Still not credited after 2 days. I am out of pocket and nobody will give me a date.
 9. I was told 5-7 working days for the refund on ORD-302806. We are well past that.
10. why does it just say pending, pending, pending. give me a date
11. why does it just say pending, pending, pending. give me a

In [31]:
my_themes = [
    "timeline_not_met",         # 1  told 5-7 days, past it
    "money_not_received",       # 2  approved, nothing arrived
    "money_not_received",       # 3  no refund, 6 days
    "no_status_visibility",     # 4  zero visibility
    "no_status_visibility",     # 5  says processing
    "no_status_visibility",     # 6  status never changes
    "no_status_visibility",     # 7  nobody can tell me when
    "money_not_received",       # 8  out of pocket
    "timeline_not_met",         # 9
    "no_status_visibility",     # 10 pending pending pending
    "no_status_visibility",     # 11
    "money_not_received",       # 12
    "money_not_received",       # 13
    "money_not_received",       # 14
    "no_status_visibility",     # 15
    "no_status_visibility",     # 16
    "no_status_visibility",     # 17
    "money_not_received",       # 18
    "no_status_visibility",     # 19
    "money_not_received",       # 20
    "timeline_not_met",         # 21
    "money_not_received",       # 22 no response prefix, but substance is refund
    "no_status_visibility",     # 23
    "timeline_not_met",         # 24
    "timeline_not_met",         # 25
    "no_status_visibility",     # 26
    "money_not_received",       # 27
    "no_status_visibility",     # 28
    "no_status_visibility",     # 29
    "no_status_visibility",     # 30
]

my_gaps = [
    True,   # 1  wants a date
    False,  # 2  approved but absent - payment may have failed
    True,   # 3
    True,   # 4
    True,   # 5
    True,   # 6
    True,   # 7
    True,   # 8
    True,   # 9
    True,   # 10
    True,   # 11
    True,   # 12
    True,   # 13
    False,  # 14 approved but absent
    True,   # 15
    True,   # 16
    True,   # 17
    False,  # 18 approved but absent
    True,   # 19
    False,  # 20 approved but absent
    True,   # 21
    True,   # 22
    True,   # 23
    True,   # 24
    True,   # 25
    True,   # 26
    True,   # 27
    True,   # 28
    True,   # 29
    True,   # 30
]

val["my_theme"] = my_themes
val["my_gap"] = my_gaps
print("labelled", len(val))

labelled 30


In [32]:
theme_acc = (val["ai_theme"] == val["my_theme"]).mean()
gap_acc   = (val["ai_gap"]   == val["my_gap"]).mean()

print(f"theme accuracy:            {theme_acc:.1%}")
print(f"self-service gap accuracy: {gap_acc:.1%}")
print()
for row in val[val["ai_theme"] != val["my_theme"]].itertuples():
    print(f"\n{row.customer_text}")
    print(f"  AI: {row.ai_theme}  |  You: {row.my_theme}")
    

theme accuracy:            86.7%
self-service gap accuracy: 90.0%


This is my second time asking about the refund for ORD-121931. Nobody can tell me when I get my money back.
  AI: money_not_received  |  You: no_status_visibility

This is my fourth time asking about the refund for ORD-120424. Nobody can tell me when I get my money back.
  AI: money_not_received  |  You: no_status_visibility

This is my fourth time asking about the refund for ORD-846963. Nobody can tell me when I get my money back.
  AI: money_not_received  |  You: no_status_visibility

Still waiting. This is my second time asking about the refund for ORD-691790. Nobody can tell me when I get my money back.
  AI: money_not_received  |  You: no_status_visibility


In [33]:
val.to_csv("data/validation_set.csv", index=False)
print("saved — theme_acc", round(theme_acc, 3), "| gap_acc", round(gap_acc, 3))

saved — theme_acc 0.867 | gap_acc 0.9


In [34]:
ai.to_csv("data/ai_classifications.csv", index=False)
print("saved", len(ai))

saved 200
